**Challenge Statement**

Managing customer support for an online car parts retailer is becoming increasingly complex and time-consuming. <br/>
The customer service team receives 100+ daily inquiries about car parts compatibility and car service appointments. <br/>
The current manual process requires to search in the database and sometimes coordination with multiple departments, leading to unpredictable response time and inconsistent customer experiences.

**Solution**

Build an automated Multi-Agent Customer Support System that handles customer inquiries end-to-end, reducing human intervention, while providing accurate, personalized responses across the board.

**Value Proposition**

This multi-agent system will reduce customer support response time from 10-30 min to under 5 minutes, decreasing operational costs by at least 50%, and improve customer satisfaction by at least 20%, through fast and accurate responses.

**Architecture**
- Two sequential specialized agents powered by an LLM.
- A2A Protocol for agent-to-agent routing.
- Two custom tools.
- Sessions & state management (InMemorySessionService).
- Observability.
- Test cases to evaluate the agents performance.

In [110]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [111]:
pip install -q google-adk[a2a]

Note: you may need to restart the kernel to use updated packages.


In [112]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


In [113]:
import json
import requests
import subprocess
import asyncio
import time
import traceback
from datetime import datetime
import uuid
import logging
import copy
from typing import Dict, Any, List, Optional
from pydantic import BaseModel, Field
from google.adk.agents import LlmAgent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.sessions import InMemorySessionService, Session
from google.adk.memory import InMemoryMemoryService
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.genai import types

# Hide additional warnings in the notebook
import warnings

warnings.filterwarnings("ignore")

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [114]:
# Define the car parts compatibility Database.

COMPATIBILITY_DB = {
    "Honda Civic 2020": ["Brake Pads Model BP-2020", "Oil Filter OF-450", "Air Filter AF-2020"],
    "Toyota Camry 2019": ["Brake Pads Model BP-1900", "Oil Filter OF-400", "Air Filter AF-1919"],
    "Ford F-150 2022": ["Brake Pads Model BP-2200", "Oil Filter OF-500", "Air Filter AF-2200"]
}

print("✅ Database created successfully.")

✅ Database created successfully.


In [115]:
# Define tool to check car part compatibility.

def check_compatibility(vehicle_info: str) -> str:
    """
    Check if parts are compatible with a specific vehicle.
    Input: Vehicle information (manufacturer, model, year).
    Output: List of compatible parts.
    """
    vehicle_info = vehicle_info.lower()

    # Query the database.
    for vehicle, parts in COMPATIBILITY_DB.items():
        # Return JSON string if a match is found.
        if vehicle_info in vehicle.lower():
            return json.dumps({
                "vehicle": vehicle,
                "compatible_parts": parts
            })

    # If no match is found.
    return json.dumps({
        "vehicle": vehicle_info,
        "compatible_parts": [],
        "message": "No compatible parts found for this vehicle."
    })

print("✅ Check Compatibility tool created!")
print(f"🤖 Test: {check_compatibility('Honda Civic 2020')}")

✅ Check Compatibility tool created!
🤖 Test: {"vehicle": "Honda Civic 2020", "compatible_parts": ["Brake Pads Model BP-2020", "Oil Filter OF-450", "Air Filter AF-2020"]}


In [116]:
# Define tool to schedule a service appointment.

def schedule_appointment(customer_info: str, service_type: str, preferred_date: str) -> str:
    """
    Schedule a service appointment for the customer.
    Input: Customer information, service type, preferred date.
    Output: Appointment confirmation.
    """
    # Simulate appointment scheduling.
    appointment_id = f"APT-{datetime.now().strftime('%Y%m%d')}-{int(time.time()) % 10000}"
    
    return json.dumps({
        "appointment_id": appointment_id,
        "customer_info": customer_info,
        "service_type": service_type,
        "scheduled_date": preferred_date,
        "confirmation": "Your appointment has been scheduled successfully."
    })

print("✅ Service Appointment tool created!")
print(f"🤖 Test: {schedule_appointment('Adrian TK', 'Engine strange noise', '01.02.2026')}")

✅ Service Appointment tool created!
🤖 Test: {"appointment_id": "APT-20251120-4802", "customer_info": "Adrian TK", "service_type": "Engine strange noise", "scheduled_date": "01.02.2026", "confirmation": "Your appointment has been scheduled successfully."}


In [117]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [118]:
# Define the Technical support agent.

technical_support_agent = LlmAgent(
    name = "technical_support_agent",
    model = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    description = "Technical support agent that provides technical guidance on car parts compatibility, installation and troubleshooting.",
    instruction = """You are a certified automotive technician with 10+ years of experience in car parts and systems.
                    You excel at explaining technical concepts in simple terms.
                    If any tool returns status "error", explain the issue to the user clearly.""",
    tools = [check_compatibility],
)

print("✅ Technical support agent created with custom function tool.")
print("🔧 Available tools:")
print("  • check_compatibility - Check the compatibility of car parts.")
print("   Model: gemini-2.5-flash-lite")
print("   Ready to help customers!")

✅ Technical support agent created with custom function tool.
🔧 Available tools:
  • check_compatibility - Check the compatibility of car parts.
   Model: gemini-2.5-flash-lite
   Ready to help customers!


In [119]:
# Define the Service Coordinator support agent.

service_coordinator_agent = LlmAgent(
    name = "service_coordinator_agent",
    model = Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    description = "Service Coordinator agent that handles all service scheduling and appointment management.",
    instruction = """You specialize in coordinating automotive services and managing customer appointments.
                    You have access to all scheduling systems..
                    If any tool returns status "error", explain the issue to the user clearly.""",
    tools = [schedule_appointment],
)

print("✅ Service Coordinator support agent created with custom function tool.")
print("🔧 Available tools:")
print("  • schedule_appointment - Coordinate and schedule car service appointments.")
print("   Model: gemini-2.5-flash-lite")
print("   Ready to help customers!")

✅ Service Coordinator support agent created with custom function tool.
🔧 Available tools:
  • schedule_appointment - Coordinate and schedule car service appointments.
   Model: gemini-2.5-flash-lite
   Ready to help customers!


In [120]:
# Logging for observability.

logging.basicConfig(level = logging.INFO) # Messages at INFO level and higher will be displayed.
logger = logging.getLogger("AutoPartsSupport") # Create a logger with the name AutoPartsSupport.

In [121]:
# Define the class template for customer requests.

class CustomerRequest(BaseModel):
    customer_id: str
    query: str
    vehicle: Optional[str] = Field(None, description = "The vehicle maker, model, and year for part compatibility.")
    service_needed: Optional[bool] = Field(False, description = "Whether the user is requesting an appointment.")
    preferred_date: Optional[str] = Field(None, description="The preferred date for the appointment.")

In [122]:
# Test cases to evaluate the agents performance.

test_cases = [
    CustomerRequest(
        customer_id = "CUST-001",
        query = "Need brake pads for my Honda Civic 2020.",
        vehicle = "Honda Civic 2020"
    ),
    CustomerRequest(
        customer_id = "CUST-002",
        query = "Schedule brake install for next Monday.",
        service_needed = True,
        preferred_date = "2025-11-24"
    ),
    CustomerRequest(
        customer_id = "CUST-003",
        query = "Check compatibility and schedule install.",
        vehicle = "Toyota Camry 2019",
        service_needed = True,
        preferred_date = "2025-11-25"
    )
]

print("✅ Test cases prepared.")

✅ Test cases prepared.


In [123]:
# Convert the the agent instances to A2A format.

a2a_technical_support_agent = to_a2a(technical_support_agent)
a2a_service_coordinator_agent = to_a2a(service_coordinator_agent)

print("✅ Agents converted to A2A format.")

✅ Agents converted to A2A format.


In [124]:
# Define the sequential chain for the agents.
# The Technical Support agent will run first.
# The Service Coordinator agent will run second, but only if a service is needed.

sequential_chain = [
    {
        "agent_name": "a2a_technical_support_agent",
        "tools": [check_compatibility],
        "condition": None  # Always runs.
    },
    {
        "agent_name": "a2a_service_coordinator_agent",
        "tools": [schedule_appointment],
        "condition": lambda ctx: ctx.get("service_needed", False)  # Runs only if service_needed=True.
    }
]

print("✅ A2A Sequential Chain defined.")

✅ A2A Sequential Chain defined.


In [125]:
# Store user session data in memory, while the app is running.

session_service = InMemorySessionService()

In [126]:
# Execute the test cases to check the agents performance. 

def run_sequential_chain_with_sessions(test_cases):
    for i, tc in enumerate(test_cases):
        customer_id = tc.customer_id
        logger.info(f"===== Running Test Case {i+1} | Customer ID: {customer_id} =====")
        logger.info(f"Query: {tc.query}")
        print("\n========================================================")
        print(f"🚀 Running Test Case {i+1} (Customer ID: {customer_id})")
        print(f"👉 Query: {tc.query}")
        print("========================================================")

        # Load or create a session for the customer.
        session: Session = session_service.get_or_create_session(customer_id)
        context = tc.model_dump() if hasattr(tc, "model_dump") else tc.dict()
        session["input"] = context  # Save input to session

        responses = []

        for step in sequential_chain:
            agent_name = step["agent_name"]
            condition = step.get("condition")
            should_run = True
            
            if condition:
                should_run = condition(session["input"])
            if not should_run:
                logger.info(f"Agent {agent_name} skipped due to condition.")
                continue

            for t in step["tools"]:
                if t.__name__ == "check_compatibility" and session["input"].get("vehicle"):
                    start_time = time.time()
                    res = t(session["input"]["vehicle"])
                    elapsed = time.time() - start_time
                    output = json.loads(res)
                    responses.append({"agent": agent_name, "tool": t.__name__, "output": output})
                    logger.info(f"{agent_name} | {t.__name__} completed in {elapsed:.2f}s")
                    logger.info(f"{agent_name} | {t.__name__} output: {json.dumps(output)}")
                    print(f"🤖 {agent_name} | {t.__name__} Output:")
                    print(json.dumps(output, indent=2))
                    
                    # Save to session.
                    session["last_compatibility"] = output

                elif t.__name__ == "schedule_appointment" and session["input"].get("service_needed"):
                    start_time = time.time()
                    res = t(session["input"]["customer_id"], session["input"]["query"], session["input"]["preferred_date"])
                    elapsed = time.time() - start_time
                    output = json.loads(res)
                    responses.append({"agent": agent_name, "tool": t.__name__, "output": output})
                    logger.info(f"{agent_name} | {t.__name__} completed in {elapsed:.2f}s")
                    logger.info(f"{agent_name} | {t.__name__} output: {json.dumps(output)}")
                    print(f"🤖 {agent_name} | {t.__name__} Output:")
                    print(json.dumps(output, indent=2))
                    # Save to session
                    session["last_appointment"] = output

        # Save final responses to session.
        session["responses"] = responses

        print("\n📌 Final Summary for Test Case:")
        print(json.dumps(responses, indent=2))
        logger.info(f"Test Case {i+1} Final Summary saved to session: {customer_id}")

# Run the sequential chain with session support.
run_sequential_chain(test_cases)


🚀 Running Test Case 1 (Customer ID: CUST-001)
👉 Query: Need brake pads for my Honda Civic 2020.
🤖 a2a_technical_support_agent | check_compatibility Output:
{
  "vehicle": "Honda Civic 2020",
  "compatible_parts": [
    "Brake Pads Model BP-2020",
    "Oil Filter OF-450",
    "Air Filter AF-2020"
  ]
}

📌 Final Summary for Test Case:
[
  {
    "agent": "a2a_technical_support_agent",
    "tool": "check_compatibility",
    "output": {
      "vehicle": "Honda Civic 2020",
      "compatible_parts": [
        "Brake Pads Model BP-2020",
        "Oil Filter OF-450",
        "Air Filter AF-2020"
      ]
    }
  }
]

🚀 Running Test Case 2 (Customer ID: CUST-002)
👉 Query: Schedule brake install for next Monday.
🤖 a2a_service_coordinator_agent | schedule_appointment Output:
{
  "appointment_id": "APT-20251120-4802",
  "customer_info": "CUST-002",
  "service_type": "Schedule brake install for next Monday.",
  "scheduled_date": "2025-11-24",
  "confirmation": "Your appointment has been scheduled s